# GPTCheMistral: Chemical Queries with a LoRA-Fine-Tuned Mistral LLM 

## Checking the GPU

In [1]:
import torch

print(f"GPUs  : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} : {name} ({vram:.0f} GB)")


GPUs  : 1
GPU 0 : Tesla T4 (15 GB)


## Install exact compatible versions of libraries

In [2]:
# ── Install exact compatible versions ───────────────────
!pip install -q huggingface_hub==0.23.4
!pip install -q transformers==4.42.0
!pip install -q bitsandbytes==0.43.1
!pip install -q peft==0.11.1
!pip install -q accelerate==0.31.0
!pip install -q tokenizers==0.19.1

print("Pinned versions installed!")

Pinned versions installed!


In [3]:
# ── Fix all version conflicts ────────────────────────────
!pip install -q -U huggingface_hub
!pip install -q -U transformers
!pip install -q -U bitsandbytes
!pip install -q -U peft
!pip install -q -U accelerate
!pip install -q -U tokenizers

print("All updated!")

# verify versions
import huggingface_hub
import transformers
import bitsandbytes
import peft

print(f"huggingface_hub : {huggingface_hub.__version__}")
print(f"transformers    : {transformers.__version__}")
print(f"bitsandbytes    : {bitsandbytes.__version__}")
print(f"peft            : {peft.__version__}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.42.0 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.11.0 which is incompatible.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.11.0 which is incompatible.
All updated!
huggingface_hub : 1.11.0
transformers    : 5.5.4
bitsandbytes    : 0.49.2
peft            : 0.19.1


In [8]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 35.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 139.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [datasets]4/5 [datasets]


## Output input directories

In [6]:
import os

DATA_FILE  = "chemistry_train.jsonl"
OUTPUT_DIR = "chem_mistral"

os.makedirs(OUTPUT_DIR, exist_ok=True)
#os.makedirs(OUTPUT_DIR2, exist_ok=True)

# verify file exists
size_mb = os.path.getsize(DATA_FILE) / (1024**2)
print(f"File found! Size : {size_mb:.1f} MB")
print(f"Output dir       : {OUTPUT_DIR}")
print("Ready!")
size_mb1 = os.path.getsize(OUTPUT_DIR)/ (1024**2)




File found! Size : 115.1 MB
Output dir       : chem_mistral
Ready!


## Loading 5 % of data for faster training

In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=DATA_FILE,
    split="train"
)

# ── use only 20% of data ──────────────────────
#dataset = dataset.select(range(min(20000, len(dataset))))
dataset = dataset.select(range(5000))
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train : {len(dataset['train']):,} entries")
print(f"Val   : {len(dataset['test']):,} entries")
print(f"\nSample:")
print(dataset['train'][0]['text'][:400])

Generating train split: 0 examples [00:00, ? examples/s]

Train : 4,750 entries
Val   : 250 entries

Sample:
### Zirconium(IV) iodide

Chemical Properties:
  Verifiedfields: changed
  Watchedfields: changed
  verifiedrevid: 470637926
  Name: Zirconium(IV) iodide
  ImageFile1: ZrI4Troyanov.tif
  ImageSize1: 275
  OtherNames: zirconium tetraiodide
  ChemSpiderID: 75903
  PubChem: 84133
  EC_number: 237-780-9
  UNII: H6NZC1810V
  InChI: 1/4HI.Zr/h4*1H;/q;;;;+4/p-4
  InChIKey: XLMQAUWIRARSJG-XBHQNQODAZ
  SMI


## Loading Tokenizer & Mistral-7B-v0.1 

In [10]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

model_name = "mistralai/Mistral-7B-v0.1"

# 4-bit config — fits T4 15GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Mistral 7B — takes 5–10 mins...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!


## Configure LoRA

In [11]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable : {trainable:,} ({100*trainable/total:.3f}%)")
print(f"Total     : {total:,}")
print("LoRA ready!")

Trainable : 13,631,488 (0.362%)
Total     : 3,765,702,656
LoRA ready!


## Tokenization

In [12]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )

print("Tokenizing data...")
tokenized = dataset.map(
    tokenize,
    batched=True,
    batch_size=500,
    remove_columns=["text"]
)
print("Tokenization done!")

Tokenizing data...


Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenization done!


## Training (fine tuning the model)

In [13]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_ratio=0.03,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",      # ← fixed typo (had backtick)
    fp16=True,
    logging_steps=10,
    save_steps=50,                   # ← save every 25 steps (very frequent)
    save_total_limit=5,              # ← keep last 5 checkpoints
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    report_to="none",
    gradient_checkpointing_kwargs={"use_reentrant": False},  # ← add this
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorForLanguageModeling(
        tokenizer, mlm=False
    ),
)

print("Training started...")
trainer.train()
#trainer.train(resume_from_checkpoint=True)  # ← ONLY THIS LINE CHANGES
print("Training complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training started...


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,0.971924,1.006790
100,0.954386,0.926454
150,0.922262,0.901402
200,0.815434,0.877317
250,0.902468,0.849290
300,0.806396,0.843588
350,0.763554,0.833157
400,0.817683,0.826092
450,0.770374,0.820235
500,0.788714,0.814422


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete!


## Saving the model

In [14]:
# ── Save Final Model ─────────────────────────────────────
import os

FINAL_DIR = f"{OUTPUT_DIR}/final"
os.makedirs(FINAL_DIR, exist_ok=True)

print("Saving model...")
model.save_pretrained(FINAL_DIR)

print("Saving tokenizer...")
tokenizer.save_pretrained(FINAL_DIR)

# verify saved files
print("\nSaved files:")
for f in os.listdir(FINAL_DIR):
    size = os.path.getsize(os.path.join(FINAL_DIR, f)) / (1024**2)
    print(f"  ✓ {f}  →  {size:.1f} MB")

print(f"\nModel saved to: {FINAL_DIR}")

Saving model...
Saving tokenizer...

Saved files:
  ✓ adapter_config.json  →  0.0 MB
  ✓ tokenizer_config.json  →  0.0 MB
  ✓ tokenizer.json  →  3.3 MB
  ✓ README.md  →  0.0 MB
  ✓ adapter_model.safetensors  →  52.0 MB

Model saved to: chem_mistral/final


## Testing the model with various temperature

In [15]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def ask_chemistry(question):
    result = pipe(
        question,
        max_new_tokens=200,
        temperature=0.2,
        do_sample=True,
        repetition_penalty=1.1
    )
    # return only the answer part
    answer = result[0]['generated_text']
    return answer

# test it
questions = [
    "What is the molecular formula of aspirin?",
    "What are the properties of caffeine?",
    "Explain the structure of glucose.",
    "What is the boiling point of ethanol?",
    "What is the CAS number of sodium chloride?",
]

print("=== Chemistry LLM ===\n")
for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_chemistry(q)}")
    print("-" * 50)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Chemistry LLM ===

Q: What is the molecular formula of aspirin?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What is the molecular formula of aspirin?

Aspirin (acetylsalicylic acid) has a molecular formula of C9H8O4.

What is the IUPAC name for aspirin?

Acetylsalicylic acid
The correct IUPAC name for aspirin is acetylsalicylic acid.

What is the image of aspirin?

Aspirin.svg

What is the CAS number for aspirin?

50-78-0

What is the UNII for aspirin?

6321LJZ6KN

What is the ATC code for aspirin?

B01

What is the Anatomical Therapeutic Chemical classification system code for aspirin?

B01

What is the ChemSpider ID for aspirin?

1047

What is the ChEMBL identifier for
--------------------------------------------------
Q: What are the properties of caffeine?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What are the properties of caffeine?

Chemical Properties:
  Verifiedfields: changed
  Watchedfields: changed
  verifiedrevid: 470612395
  ImageFileL1: Caffeine.svg
  ImageSizeL1: 120px
  ImageAltL1: Skeletal formula of caffeine
  ImageFileR1: Caffeine-3D-balls.png
  ImageSizeR1: 120px
  ImageAltR1: Ball and stick model of the caffeine molecule
  PIN: 1,3,7-Trimethylxanthine
  OtherNames: 1,3,7-Trimethylpurine-2,6-dione; 1,3,7-Trimethyl-2,6-dioxopurin-8(7H)-one;
--------------------------------------------------
Q: Explain the structure of glucose.


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Explain the structure of glucose.

Glucose is a monosaccharide, which means it has one sugar unit. It is also an aldopentose, which means it has five carbon atoms in its chain. The molecular formula for glucose is C6H12O6.

The IUPAC name for glucose is 2,3,4,5,6-pentahydroxyhexanal.

The skeletal formula for glucose is:

The Haworth projection for glucose is:

The Fischer projection for glucose is:

The space filling model for glucose is:
--------------------------------------------------
Q: What is the boiling point of ethanol?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What is the boiling point of ethanol?

Ethanol has a boiling point of 78.3°C at standard pressure.

What is the freezing point of ethanol?

The freezing point of ethanol is -114.2°C.

What is the melting point of ethanol?

The melting point of ethanol is -114.2°C.

What is the density of ethanol?

The density of ethanol is 0.789 g/cm3.

What is the vapor pressure of ethanol?

The vapor pressure of ethanol is 61.5 kPa at 25°C.

What is the solubility of ethanol in water?

The solubility of ethanol in water is 70 g/L at 25°C.

--------------------------------------------------
Q: What is the CAS number of sodium chloride?
A: What is the CAS number of sodium chloride?

Sodium chloride (NaCl), also known as table salt or halite, is an ionic compound with the formula Na+/Cl−. It is a colorless crystalline solid that dissolves easily in water to produce a solution called brine. The sodium and chloride ions are held together by strong electrostatic forces. Sodium chloride is one of the mos

In [16]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def ask_chemistry(question):
    result = pipe(
        question,
        max_new_tokens=200,
        temperature=0.8,
        do_sample=True,
        repetition_penalty=1.1
    )
    # return only the answer part
    answer = result[0]['generated_text']
    return answer

# test it
questions = [
    "What is the molecular formula of aspirin?",
    "What are the properties of caffeine?",
    "Explain the structure of glucose.",
    "What is the boiling point of ethanol?",
    "What is the CAS number of sodium chloride?",
]

print("=== Chemistry LLM ===\n")
for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_chemistry(q)}")
    print("-" * 50)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Chemistry LLM ===

Q: What is the molecular formula of aspirin?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What is the molecular formula of aspirin?  Acetylsalicylic acid, ASA.

What are the properties of aspirin?  Aspirin is a white crystalline powder with salty taste. It dissolves slightly in water and very slowly in alcohol.

What is the melting point of aspirin?  Aspirin has two polymorphic forms. The "slow-melting form" has a melting point of 156 °C and the "fast-melting form" a melting point of 134 °C. In aqueous solution at pH 7, the ions sodium acetate and hydrogen sulfate are found in approximate equal quantities.

What is the solubility of aspirin?  Aspirin is almost insoluble in water (0.52 g/L at 25 °C). The solubility increases to 18.7 g/
--------------------------------------------------
Q: What are the properties of caffeine?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What are the properties of caffeine?  Verified fields: changed
 verifiedrevid: 470626912
Chemical name: (R,R)-1,3,7-trimethylpurine-2,6-dione
ImageFile: Caffein.svg
ImageSize: 250px
ImageName: Stereo skeletal formula of caffeine
IUPACName: (R,R)-1,3,7-trimethylpurine-2,6-dione
SMILES: [H][C@]1(C)CN[C@@]2(/C=C/C)C[C@@]([O-])([N+](=O)[C@@H]3C[C@@H]2C[C@H]([O-])[C@]13CC)(C)CO
CanonicalSMILES: C\C=
--------------------------------------------------
Q: Explain the structure of glucose.


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Explain the structure of glucose.

Glucose (C6H12O6) is a hexahydroxyhexanal with the systematic name 2,3,4,5,6-pentahydroxy-1(hydroxymethyl)-1-(hydroxymethyl)furan-3-carbaldehyde. For the structure of glucose see glycone. The ring carbon atoms are numbered from 1' to 5' anticlockwise.

What is the structure of glyceraldehyde?

Glyceraldehyde has 6 carbon atoms, and 0 double bonds. It is in the form of a triol. Glyceraldehyde is tautomerized into glycerol or alditriose.

What is the molecular formula for dihydroxyacetone?

Dihydroxyacetone is an
--------------------------------------------------
Q: What is the boiling point of ethanol?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What is the boiling point of ethanol?

Ethanol has a boiling point of 78.3°C (172.94°F) at a pressure of 101.325 kPa (1 atm). At 25°C, the vapour pressure of ethyl alcohol is 50 mbar. This corresponds to a molar Q-value of 338.6 J/mol K and a Celsius Q-value of 307. Cethyl alcohol has a boiling point of 179.96 °C at 1 Atmosphere. Ethanol has a melting point of 114.1 °C, and a freezing point of 16.5 °C (61.7 °F).

What is the boiling point of acetic acid?

Acetic acid has a boiling point of 118.1
--------------------------------------------------
Q: What is the CAS number of sodium chloride?
A: What is the CAS number of sodium chloride?

2017-04-25T18:09:36Z (UTC)
Sodium chloride IUPAC International Union of Pure and Applied Chemistry, 1992
Sodium chloride InChI=1/Na.ClH.O4/c1*1-2;/h1*1H;/q+1;;;;;-2
Sodium chloride InChIKey=JCQQCWXVYDVFET-UHFFFAOYA/S=C(=O)/O[Na+]([Cl-])=O
147-96-3 CAS Chemical Abstracts Service, data from QCICAS2018 database
14696-27-6 DIN Chemisches Zentralblatt, da

In [17]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def ask_chemistry(question):
    result = pipe(
        question,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=True,
        repetition_penalty=1.1
    )
    # return only the answer part
    answer = result[0]['generated_text']
    return answer

# test it
questions = [
    "What is the molecular formula of aspirin?",
    "What are the properties of caffeine?",
    "Explain the structure of glucose.",
    "What is the boiling point of ethanol?",
    "What is the CAS number of sodium chloride?",
]

print("=== Chemistry LLM ===\n")
for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_chemistry(q)}")
    print("-" * 50)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Chemistry LLM ===

Q: What is the molecular formula of aspirin?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What is the molecular formula of aspirin?

Aspirin (acetylsalicylic acid) has a chemical formula of C9H8O4.

What is the IUPAC name for aspirin?

Acetylsalicylic acid
The IUPAC name for aspirin is 2-(acetyloxy)benzoic acid.

What is the CAS number for aspirin?

76-68-5
Chemical Properties: Verified fields: changed
Watchedfields: changed
 verifiedrevid: 401373893
Aspirin (acetylsalicylic acid) has a CAS number of 76-68-5.

What is the SMILES notation for aspirin?

O=C(Oc1ccc(Cl)cc1)(C)C(=O)O
The SMILES notation for aspirin
--------------------------------------------------
Q: What are the properties of caffeine?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What are the properties of caffeine?  Verified fields: changed

Chemical structure:
  verifiedrevid: 470615239
  IUPAC_name: 1,3,7-Trimethylpurine-2,6-dione
  image: Caffeine.svg
  image_class: skin-invert-image
  alt: Skeletal formula of caffeine
  width: 180px
  image2: Caffeine molecule ball.png
  image_class2: bg-transparent
  alt2: Ball-and-stick model of the caffeine molecule
  width2: 180px
  tradename: Cafix, Caffix, Caffixx, Caffixxx, Caffixxxxx, Caffixxxxxx, Caffixxxxxxx, Caffixxxxxxx
--------------------------------------------------
Q: Explain the structure of glucose.


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Explain the structure of glucose.

Glucose is a monosaccharide with the molecular formula C6H12O6. It has six carbon atoms, twelve hydrogen atoms and six oxygen atoms. The IUPAC name for glucose is 2,3,4,5,6-pentahydroxyhexanal. Glucose is also known as dextrose or grape sugar.

Explain the structure of fructose.

Fructose is a ketose monosaccharide with the molecular formula C6H12O6. It has six carbon atoms, twelve hydrogen atoms and six oxygen atoms. The IUPAC name for fructose is 2,5-dihydroxyhexan-1-ulose. Fructose is also known as levulose or fruit sugar.

Explain the structure of sucrose.

Sucrose is a disaccharide with the molecular
--------------------------------------------------
Q: What is the boiling point of ethanol?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: What is the boiling point of ethanol?

The boiling point of ethanol is 78.3°C at a pressure of 101.3 kPa (760 mmHg). The boiling point of ethanol is 94.4°C at a pressure of 1 atm.

What is the boiling point of ethanol in Celsius?

The boiling point of ethanol is 78.3°C at a pressure of 101.3 kPa (760 mmHg). The boiling point of ethanol is 94.4°C at a pressure of 1 atm.

What is the boiling point of ethanol in Fahrenheit?

The boiling point of ethanol is 175.2°F at a pressure of 101.3 kPa (760 mmHg).
--------------------------------------------------
Q: What is the CAS number of sodium chloride?
A: What is the CAS number of sodium chloride?

Sodium chloride (NaCl) is a chemical compound with the formula Na+ and Cl− ions. It is commonly known as table salt or halite, and is one of the most important dietary seasonings. Sodium chloride is also used in many industrial applications.

What is the molecular weight of sodium chloride?

The molecular weight of sodium chloride is 58.44 g/mol.